# **Regression**

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import numpy as np
import seaborn as sns

In [3]:
file_name= "..\\dm1_25_26_dataset\\cleaned_dataset.csv"
df = pd.read_csv(file_name)

In [4]:
df

,BGGId,Name,YearPublished,GameWeight,MinPlayers,MaxPlayers,ComAgeRec,ComMinPlaytime,ComMaxPlaytime,IsReimplementation,...,Score_cgs,Score_wargames,Score_partygames,Score_childrensgames,Rating,Avg_popularity_log,NumAlternates_log,NumImplementations_log,NumExpansions_log,NumWeightVotes_log
0,140386,Assassin's Creed: Arena,2014,1.8333,2,4,8.0,30.0,30.0,0,...,0.0,0.0,0.0,0.0,1,5.299983,0.000000,0.0,0.693147,1.945910
1,344114,Bag of Chips,2021,1.0000,2,5,8.0,15.0,20.0,0,...,0.0,0.0,0.0,0.0,2,3.799228,0.693147,0.0,0.000000,1.945910
2,319196,Gùgōng: Deluxe Big Box,2020,3.6667,1,5,10.0,30.0,150.0,0,...,0.0,0.0,0.0,0.0,3,5.744071,0.000000,0.0,0.693147,2.302585
3,11404,LetterFlip,2004,1.3077,2,2,7.0,20.0,20.0,0,...,0.0,0.0,0.0,0.0,1,4.013375,0.000000,0.0,0.000000,2.639057
4,281020,Treasures of Cibola,2019,1.5000,2,4,10.0,15.0,20.0,0,...,0.0,0.0,0.0,0.0,2,4.422849,1.386294,0.0,0.000000,1.098612
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
19724,3233,Ballast,1998,1.0000,2,4,8.0,15.0,15.0,0,...,0.0,0.0,0.0,0.0,1,3.721669,0.000000,0.0,0.000000,1.609438
19725,13359,Leapfrog,2004,1.5000,1,6,6.0,20.0,20.0,0,...,0.0,0.0,0.0,0.0,2,4.660289,1.098612,0.0,0.000000,2.833213
19726,3295,Athos,1993,1.6667,2,4,11.0,45.0,45.0,0,...,0.0,0.0,0.0,0.0,1,4.426841,0.000000,0.0,0.000000,1.386294
19727,176524,Hoplomachus: Origins,2015,3.0370,1,2,14.0,15.0,30.0,0,...,0.0,0.0,0.0,0.0,3,6.454150,0.000000,0.0,2.484907,3.332205


### **Computing correlation**

In [5]:
# compute correlation between ComAgeRec, GameWeight, ComMaxPlaytime and NumExpansions_log
correlation_matrix = df[['ComAgeRec', 'GameWeight', 'ComMaxPlaytime', 'NumExpansions_log','NumAlternates_log','Avg_popularity_log']].corr()
print(correlation_matrix)

                    ComAgeRec  GameWeight  ComMaxPlaytime  NumExpansions_log  \
ComAgeRec            1.000000    0.587507        0.546239           0.122082   
GameWeight           0.587507    1.000000        0.652666           0.268047   
ComMaxPlaytime       0.546239    0.652666        1.000000           0.120900   
NumExpansions_log    0.122082    0.268047        0.120900           1.000000   
NumAlternates_log   -0.241481   -0.161087       -0.142595           0.100882   
Avg_popularity_log   0.076716    0.250388        0.131464           0.436529   

                    NumAlternates_log  Avg_popularity_log  
ComAgeRec                   -0.241481            0.076716  
GameWeight                  -0.161087            0.250388  
ComMaxPlaytime              -0.142595            0.131464  
NumExpansions_log            0.100882            0.436529  
NumAlternates_log            1.000000            0.414653  
Avg_popularity_log           0.414653            1.000000  


### **Variable definition** + **Train/test split** 

In [11]:
# 1. Define Target Variables
target_w = 'GameWeight'           # For Univariate and Multiple
target_p = 'Avg_popularity_log'   # For Multivariate (second target)
multivariate_targets = [target_w, target_p]

# 2. Define Feature Sets
# Univariate: One feature to predict weight
features_uni = ['ComAgeRec']

# Multiple: Original 3 features to predict weight
features_multiple = ['ComAgeRec', 'ComMaxPlaytime','NumExpansions_log']

# Multivariate: Optimized features to predict both weight and popularity
features_multivariate = ['NumAlternates_log', 'ComMaxPlaytime', 'NumExpansions_log']

# 3. Combine all necessary columns for a unified split
all_features = list(set(features_uni + features_multiple + features_multivariate))
all_targets = list(set([target_w, target_p]))

# 4. Perform a single split to ensure all models use the same data samples
X_train_full, X_test_full, y_train_full, y_test_full = train_test_split(
    df[all_features], 
    df[all_targets], 
    test_size=0.2, 
    random_state=42
)

# 5. Create specific subsets for each analysis type

# Subsets for Univariate Regression (Target: GameWeight)
X_train_uni = X_train_full[features_uni]
X_test_uni = X_test_full[features_uni]
y_train_uni = y_train_full[target_w]
y_test_uni = y_test_full[target_w]

# Subsets for Multiple Regression (Target: GameWeight)
X_train_multi = X_train_full[features_multiple]
X_test_multi = X_test_full[features_multiple]
y_train_multi = y_train_full[target_w]
y_test_multi = y_test_full[target_w]

# Subsets for Multivariate Regression (Targets: GameWeight AND Avg_popularity_log)
# Note: y contains both columns as a matrix
X_train_mvar = X_train_full[features_multivariate]
X_test_mvar = X_test_full[features_multivariate]
y_train_mvar = y_train_full[multivariate_targets]
y_test_mvar = y_test_full[multivariate_targets]

print("Split successful!")
print(f"Total dataset size: {len(df)}")
print(f"Features used in split: {all_features}")
print(f"Targets for Multivariate: {multivariate_targets}")

Split successful!
Total dataset size: 19729
Features used in split: ['ComAgeRec', 'NumAlternates_log', 'ComMaxPlaytime', 'NumExpansions_log']
Targets for Multivariate: ['GameWeight', 'Avg_popularity_log']


### **Evaluation function**

In [12]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "R2": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "MSE": mean_squared_error(y_test, y_pred)
    }

### **Univariate regression** 

In [13]:
# --- STEP 2: UNIVARIATE REGRESSION (Target: GameWeight, Feature: ComAgeRec) ---

# A. Linear Regression (No hyperparameters to optimize)
lin_uni = LinearRegression()
lin_uni.fit(X_train_uni, y_train_uni)
res_lin_uni = evaluate_model(lin_uni, X_test_uni, y_test_uni)
best_params_lin_uni = "N/A"

# B. Ridge Regression (L2 Regularization)
ridge_uni_search = GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}, scoring="neg_mean_squared_error", cv=5)
ridge_uni_search.fit(X_train_uni, y_train_uni)
best_params_ridge_uni = ridge_uni_search.best_params_
res_ridge_uni = evaluate_model(ridge_uni_search.best_estimator_, X_test_uni, y_test_uni)

# C. Lasso Regression (L1 Regularization)
lasso_uni_search = GridSearchCV(Lasso(max_iter=10000), {"alpha": [0.001, 0.01, 0.1, 1, 10]}, scoring="neg_mean_squared_error", cv=5)
lasso_uni_search.fit(X_train_uni, y_train_uni)
best_params_lasso_uni = lasso_uni_search.best_params_
res_lasso_uni = evaluate_model(lasso_uni_search.best_estimator_, X_test_uni, y_test_uni)

# D. Decision Tree (Non-linear depth optimization)
dt_uni_search = GridSearchCV(DecisionTreeRegressor(random_state=42), {"max_depth": [2, 4, 8, 16, 32]}, scoring="neg_mean_squared_error", cv=5)
dt_uni_search.fit(X_train_uni, y_train_uni)
best_params_dt_uni = dt_uni_search.best_params_
res_dt_uni = evaluate_model(dt_uni_search.best_estimator_, X_test_uni, y_test_uni)

# E. K-Nearest Neighbors (KNN - Optimized for neighbors)
knn_uni_search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]), 
                              {"knn__n_neighbors": [3, 5, 10, 20, 40, 50, 60]}, 
                              scoring="neg_mean_squared_error", cv=5)
knn_uni_search.fit(X_train_uni, y_train_uni)
best_params_knn_uni = knn_uni_search.best_params_
res_knn_uni = evaluate_model(knn_uni_search.best_estimator_, X_test_uni, y_test_uni)

# F. Store Results in DataFrame with clear parameter strings
results_uni = pd.DataFrame([
    {"Model": "Linear", "Task": "Univariate", **res_lin_uni, "Best Params": best_params_lin_uni},
    {"Model": "Ridge", "Task": "Univariate", **res_ridge_uni, "Best Params": str(best_params_ridge_uni)},
    {"Model": "Lasso", "Task": "Univariate", **res_lasso_uni, "Best Params": str(best_params_lasso_uni)},
    {"Model": "Decision Tree", "Task": "Univariate", **res_dt_uni, "Best Params": str(best_params_dt_uni)},
    {"Model": "K-NN", "Task": "Univariate", **res_knn_uni, "Best Params": str(best_params_knn_uni)},
])

print("\n--- Hyperparameter Optimization Summary (Univariate) ---")
print(f"Optimal Ridge Alpha: {best_params_ridge_uni.get('alpha')}")
print(f"Optimal Lasso Alpha: {best_params_lasso_uni.get('alpha')}")
print(f"Optimal DT Max Depth: {best_params_dt_uni.get('max_depth')}")
print(f"Optimal KNN Neighbors: {best_params_knn_uni.get('knn__n_neighbors')}")
print("-" * 55)

display(results_uni)


--- Hyperparameter Optimization Summary (Univariate) ---
Optimal Ridge Alpha: 10
Optimal Lasso Alpha: 0.001
Optimal DT Max Depth: 8
Optimal KNN Neighbors: 60
-------------------------------------------------------


,Model,Task,R2,MAE,MSE,Best Params
0,Linear,Univariate,0.331509,0.467048,0.376091,N/A
1,Ridge,Univariate,0.331512,0.467049,0.376090,{'alpha': 10}
2,Lasso,Univariate,0.331533,0.467062,0.376078,{'alpha': 0.001}
3,Decision Tree,Univariate,0.385521,0.452954,0.345705,{'max_depth': 8}
4,K-NN,Univariate,0.377553,0.457599,0.350188,{'knn__n_neighbors': 60}


### **Multiple regression** 

In [14]:
# --- STEP 3: MULTIPLE REGRESSION (Target: GameWeight, Features: Age + Time + Expansions) ---

# A. Linear Regression (No hyperparameters to optimize)
lin_multi = LinearRegression()
lin_multi.fit(X_train_multi, y_train_multi)
res_lin_multi = evaluate_model(lin_multi, X_test_multi, y_test_multi)
best_params_lin_multi = "N/A"

# B. Ridge Regression
ridge_multi_search = GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}, scoring="neg_mean_squared_error", cv=5)
ridge_multi_search.fit(X_train_multi, y_train_multi)
res_ridge_multi = evaluate_model(ridge_multi_search.best_estimator_, X_test_multi, y_test_multi)
best_params_ridge_multi = ridge_multi_search.best_params_

# C. Lasso Regression
lasso_multi_search = GridSearchCV(Lasso(max_iter=10000), {"alpha": [0.001, 0.01, 0.1, 1, 10]}, scoring="neg_mean_squared_error", cv=5)
lasso_multi_search.fit(X_train_multi, y_train_multi)
res_lasso_multi = evaluate_model(lasso_multi_search.best_estimator_, X_test_multi, y_test_multi)
best_params_lasso_multi = lasso_multi_search.best_params_

# D. Decision Tree
dt_multi_search = GridSearchCV(DecisionTreeRegressor(random_state=42), {"max_depth": [2, 4, 8, 16, 32]}, scoring="neg_mean_squared_error", cv=5)
dt_multi_search.fit(X_train_multi, y_train_multi)
res_dt_multi = evaluate_model(dt_multi_search.best_estimator_, X_test_multi, y_test_multi)
best_params_dt_multi = dt_multi_search.best_params_

# E. K-Nearest Neighbors (KNN)
# Using pipeline for proper scaling during cross-validation
knn_multi_search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]), 
                                {"knn__n_neighbors": [3, 5, 10, 20, 40, 50, 60]}, 
                                scoring="neg_mean_squared_error", cv=5)
knn_multi_search.fit(X_train_multi, y_train_multi)
res_knn_multi = evaluate_model(knn_multi_search.best_estimator_, X_test_multi, y_test_multi)
best_params_knn_multi = knn_multi_search.best_params_

# F. Store Results in DataFrame (Including the best hyperparameters as strings)
results_multi = pd.DataFrame([
    {"Model": "Linear", "Task": "Multiple", **res_lin_multi, "Best Params": best_params_lin_multi},
    {"Model": "Ridge", "Task": "Multiple", **res_ridge_multi, "Best Params": str(best_params_ridge_multi)},
    {"Model": "Lasso", "Task": "Multiple", **res_lasso_multi, "Best Params": str(best_params_lasso_multi)},
    {"Model": "Decision Tree", "Task": "Multiple", **res_dt_multi, "Best Params": str(best_params_dt_multi)},
    {"Model": "K-NN", "Task": "Multiple", **res_knn_multi, "Best Params": str(best_params_knn_multi)},
])

print("\n--- Hyperparameter Optimization Summary (Multiple Regression) ---")
print(f"Optimal Ridge Alpha: {best_params_ridge_multi.get('alpha')}")
print(f"Optimal Lasso Alpha: {best_params_lasso_multi.get('alpha')}")
print(f"Optimal DT Max Depth: {best_params_dt_multi.get('max_depth')}")
print(f"Optimal KNN Neighbors: {best_params_knn_multi.get('knn__n_neighbors')}")
print("-" * 60)

display(results_multi)


--- Hyperparameter Optimization Summary (Multiple Regression) ---
Optimal Ridge Alpha: 10
Optimal Lasso Alpha: 0.001
Optimal DT Max Depth: 8
Optimal KNN Neighbors: 60
------------------------------------------------------------


,Model,Task,R2,MAE,MSE,Best Params
0,Linear,Multiple,0.516389,0.402129,0.272079,N/A
1,Ridge,Multiple,0.516386,0.402134,0.272080,{'alpha': 10}
2,Lasso,Multiple,0.516370,0.402186,0.272089,{'alpha': 0.001}
3,Decision Tree,Multiple,0.554094,0.379998,0.250866,{'max_depth': 8}
4,K-NN,Multiple,0.567885,0.376594,0.243107,{'knn__n_neighbors': 60}


### **Multivariate regression**

In [15]:
# --- STEP 4: MULTIVARIATE REGRESSION (Target Breakdown: Weight vs. Popularity) ---

# Define targets for clarity in output
multivariate_targets = [target_w, target_p] # ['GameWeight', 'Avg_popularity_log']

# Helper function to evaluate each target separately for a multivariate model
def evaluate_multivariate(model, X_test, y_test, targets):
    predictions = model.predict(X_test)
    scores = []
    
    for i, target_name in enumerate(targets):
        # Extract true values and predictions for the specific target
        y_true_target = y_test.iloc[:, i]
        y_pred_target = predictions[:, i]
        
        # Calculate metrics
        r2 = r2_score(y_true_target, y_pred_target)
        mae = mean_absolute_error(y_true_target, y_pred_target)
        mse = mean_squared_error(y_true_target, y_pred_target)
        
        scores.append({
            "Target": target_name,
            "R2": r2,
            "MAE": mae,
            "MSE": mse
        })
    return scores

# Initialize list for detailed results
mvar_detailed_results = []

# A. Linear Regression
lin_mvar = LinearRegression()
lin_mvar.fit(X_train_mvar, y_train_mvar)
for res in evaluate_multivariate(lin_mvar, X_test_mvar, y_test_mvar, multivariate_targets):
    mvar_detailed_results.append({"Model": "Linear", **res, "Best Params": "N/A"})

# B. Ridge Regression
ridge_mvar_search = GridSearchCV(Ridge(), {"alpha": [0.01, 0.1, 1, 10, 100]}, scoring="neg_mean_squared_error", cv=5)
ridge_mvar_search.fit(X_train_mvar, y_train_mvar)
for res in evaluate_multivariate(ridge_mvar_search.best_estimator_, X_test_mvar, y_test_mvar, multivariate_targets):
    mvar_detailed_results.append({"Model": "Ridge", **res, "Best Params": str(ridge_mvar_search.best_params_)})

# C. Lasso Regression
lasso_mvar_search = GridSearchCV(Lasso(max_iter=10000), {"alpha": [0.001, 0.01, 0.1, 1, 10]}, scoring="neg_mean_squared_error", cv=5)
lasso_mvar_search.fit(X_train_mvar, y_train_mvar)
for res in evaluate_multivariate(lasso_mvar_search.best_estimator_, X_test_mvar, y_test_mvar, multivariate_targets):
    mvar_detailed_results.append({"Model": "Lasso", **res, "Best Params": str(lasso_mvar_search.best_params_)})

# D. Decision Tree
dt_mvar_search = GridSearchCV(DecisionTreeRegressor(random_state=42), {"max_depth": [2, 4, 8, 16, 32]}, scoring="neg_mean_squared_error", cv=5)
dt_mvar_search.fit(X_train_mvar, y_train_mvar)
for res in evaluate_multivariate(dt_mvar_search.best_estimator_, X_test_mvar, y_test_mvar, multivariate_targets):
    mvar_detailed_results.append({"Model": "Decision Tree", **res, "Best Params": str(dt_mvar_search.best_params_)})

# E. K-Nearest Neighbors (KNN)
knn_mvar_search = GridSearchCV(Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]), 
                                {"knn__n_neighbors": [3, 5, 10, 20, 40, 50, 60]}, 
                                scoring="neg_mean_squared_error", cv=5)
knn_mvar_search.fit(X_train_mvar, y_train_mvar)
for res in evaluate_multivariate(knn_mvar_search.best_estimator_, X_test_mvar, y_test_mvar, multivariate_targets):
    mvar_detailed_results.append({"Model": "K-NN", **res, "Best Params": str(knn_mvar_search.best_params_)})

# F. Final Detailed Results DataFrame
results_mvar_separated = pd.DataFrame(mvar_detailed_results)

# Sorting by Target and then R2 for better readability
results_mvar_separated = results_mvar_separated.sort_values(by=["Target", "R2"], ascending=[True, False])

display(results_mvar_separated)

,Model,Target,R2,MAE,MSE,Best Params
9,K-NN,Avg_popularity_log,0.380870,0.833890,1.112300,{'knn__n_neighbors': 60}
7,Decision Tree,Avg_popularity_log,0.369181,0.841917,1.133302,{'max_depth': 8}
5,Lasso,Avg_popularity_log,0.348854,0.860110,1.169819,{'alpha': 0.001}
3,Ridge,Avg_popularity_log,0.348809,0.860040,1.169901,{'alpha': 0.1}
1,Linear,Avg_popularity_log,0.348809,0.860039,1.169901,N/A
6,Decision Tree,GameWeight,0.504546,0.411287,0.278741,{'max_depth': 8}
8,K-NN,GameWeight,0.503254,0.412215,0.279468,{'knn__n_neighbors': 60}
0,Linear,GameWeight,0.456958,0.437554,0.305515,N/A
2,Ridge,GameWeight,0.456958,0.437554,0.305515,{'alpha': 0.1}
4,Lasso,GameWeight,0.456946,0.437617,0.305521,{'alpha': 0.001}
